In [1]:
import pandas as pd
from sqlalchemy import create_engine

In [2]:
# 1. Paramètres de connexion MySQL
user = "root"
password = "YOODAAV"
host = "localhost"
port = "3306"
database = "Pizza"

# 2. Créer l'engine SQLAlchemy (ici avec pymysql)
engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}:{port}/{database}")

In [3]:
# 3. Charger les CSV avec pandas
df = pd.read_csv("pizza_sales.csv")

In [4]:
from sqlalchemy import text

# Ensure the target database exists (create if missing)
admin_engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}:{port}", isolation_level="AUTOCOMMIT")
with admin_engine.connect() as conn:
    exists = conn.execute(text("SELECT SCHEMA_NAME FROM INFORMATION_SCHEMA.SCHEMATA WHERE SCHEMA_NAME = :db"), {"db": database}).scalar() is not None
    if not exists:
        conn.execute(text(f'CREATE DATABASE IF NOT EXISTS `{database}`'))
        print(f'Created database "{database}"')
    else:
        print(f'Database "{database}" exists')

admin_engine.dispose()

# Recreate engine for the target database (overwrite existing engine variable)
engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}:{port}/{database}")

# Ingest with safer error decoding to avoid UnicodeDecodeError when printing exceptions
try:
    
    df.to_sql("pizza_sales", engine, if_exists="replace", index=False, method="multi", chunksize=1000)
    print("Ingestion terminée avec succès !")
except Exception as e:
    parts = []
    for a in getattr(e, "args", ()):
        if isinstance(a, bytes):
            parts.append(a.decode("utf-8", errors="replace"))
        else:
            parts.append(str(a))
    print("Erreur lors de l'ingestion :", " | ".join(parts))

Created database "Pizza"
Ingestion terminée avec succès !
